In [ ]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import json
import tiktoken

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/headlines_prediction")
# REPO_DIR = "."
os.chdir(REPO_DIR)

load_dotenv(os.path.join(REPO_DIR, ".env"), override=True)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PER_BATCH_LIMIT = 50e3 # up to 50,000 requests per batch


In [115]:
def create_embed_requests_generic(df, text_col_name, id_col_name, out_dir, embedding_model="text-embedding-3-small"):

    df = df[[id_col_name, text_col_name]].drop_duplicates().reset_index(drop=True)

    part = 0
    batches = []
    requests = []

    for i, row in df.iterrows():
        requests.append({
            "custom_id": str(row[id_col_name]),
            "method": "POST",
            "url": "/v1/embeddings",
            "body": {
                "input": row[text_col_name],
                "model": embedding_model
            }
        })

        # print(i)
        if ((len(requests)==PER_BATCH_LIMIT) | (i==(len(df)-1))):
            part = part + 1

            col_path = os.path.join(out_dir, f"requests_{text_col_name}_part{part}.jsonl")
            with open(col_path, "w") as f:
                for request in requests:
                    f.write(json.dumps(request) + "\n")
                print(f"Saved {os.path.basename(col_path)}, n = {len(requests)}, at {os.path.dirname(col_path)}")
            requests = []
            batches.append({
                'file': col_path,
                'part': part,
                'col_name': text_col_name
            })

    return(pd.json_normalize(batches))

def create_embed_requests(df, out_dir, embedding_model="text-embedding-3-small"):
    batches = []
    batches.append(create_embed_requests_generic(df, "headline_clean", "headline_id", out_dir, embedding_model))
    batches.append(create_embed_requests_generic(df, "headline_llm_clean", "id", out_dir, embedding_model))
    batches = pd.concat(batches, ignore_index=True)
    return(batches)

In [116]:
def query_embeddings(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)

    batches_id = []
    for _, batch in batches.iterrows():
        batch_input_file = client.files.create(
            file = open(batch['file'], "rb"),
            purpose = "batch"
        )

        new_batch = client.batches.create(
            input_file_id = batch_input_file.id,
            endpoint = "/v1/embeddings",
            completion_window = "24h",
            metadata = {"description": f"{os.path.basename(batch['file'])}"}
        )

        batches_id.append(new_batch.id)

    batches['id'] = batches_id
    return(batches)

In [117]:
# Check status of all batches
def check_batches_status(batches):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        file = batch['file']
        id = batch['id']
        batch_status = client.batches.retrieve(id)
        print(f"{os.path.basename(file):>52s}: {batch_status.status}")


In [129]:
def download_batched_responses(batches, out_dir):
    client = OpenAI(api_key=OPENAI_API_KEY)
    for _, batch in batches.iterrows():
        batch_id = batch['id']
        part = batch['part']
        col_name = batch['col_name']
        responses_batched_path = os.path.join(out_dir, f"responses_{col_name}_part{part}.jsonl")

        batch_status = client.batches.retrieve(batch_id)
        if batch_status.status != "completed":
            print(f"Skipping incomplete file = {os.path.basename(responses_batched_path)}, Batch ID = {batch_id}")
            continue
        
        output_file_id = batch_status.output_file_id
        responses_batched = client.files.content(output_file_id)
        responses_batched.write_to_file(responses_batched_path)
        print(f"Saved {os.path.basename(responses_batched_path)} at {os.path.dirname(responses_batched_path)}")

In [119]:
def count_tokens(text, model="text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    n_tokens = len(encoding.encode(text)) 
    return(n_tokens)

def estimate_cost(text, model="text-embedding-3-small", batched=True):
    n_tokens = np.zeros_like(text)
    for i in range(len(text)):
        n_tokens[i] = count_tokens(text[i], model)
    
    # Cost without using Batch API
    embed_token_cost = {
        'text-embedding-3-small': 0.020/1e6,
        'text-embedding-3-large': 0.130/1e6,
        'ada v2': 0.100/1e6
    }

    total_cost  = (n_tokens * embed_token_cost[model]).sum()
    if batched:
        total_cost = total_cost/2
    return total_cost

In [ ]:
data_dir = os.path.join(REPO_DIR, "Data")
temp_dir = os.path.join(REPO_DIR, "Temp/Embeddings")

headlines_completion = pd.read_csv(os.path.join(data_dir, "headlines_completion.csv"))

In [121]:
cost_description = estimate_cost(headlines_completion["headline_clean"].unique(), batched=True)
print(f"Estimated cost to embed headline using Batch API is ${cost_description:.2f}")

cost_description_llm = estimate_cost(headlines_completion["headline_llm_clean"], batched=True)
print(f"Estimated cost to embed headline_llm using Batch API is ${cost_description_llm:.2f}")

Estimated cost to embed headline using Batch API is $0.00
Estimated cost to embed headline_llm using Batch API is $0.01


In [ ]:
requests_dir = os.path.join(temp_dir, "Requests")
os.makedirs(requests_dir, exist_ok=True)

batches = create_embed_requests(headlines_completion, requests_dir, embedding_model="text-embedding-3-small")
batches_path = os.path.join(temp_dir, "batches.csv")
batches.to_csv(batches_path, index=False)
print(f"Created batched prompts with batch details stored at {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Saved requests_headline_clean_part1.jsonl, n = 10000, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Requests
Saved requests_headline_llm_clean_part1.jsonl, n = 50000, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Requests
Saved requests_headline_llm_clean_part2.jsonl, n = 9997, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Requests
Created batched prompts with batch details stored at batches.csv, n = 3, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings


In [123]:
batches = query_embeddings(batches)
batches.to_csv(batches_path, index=False)
print(f"Added batch_id to {os.path.basename(batches_path)}, n = {len(batches)}, at {os.path.dirname(batches_path)}")

Added batch_id to batches.csv, n = 3, at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings


In [ ]:
batches = pd.read_csv(os.path.join(temp_dir, "batches.csv"))
check_batches_status(batches)

                 requests_headline_clean_part1.jsonl: completed
             requests_headline_llm_clean_part1.jsonl: completed
             requests_headline_llm_clean_part2.jsonl: completed


In [ ]:
batches = pd.read_csv(os.path.join(temp_dir, "batches.csv"))
responses_dir = os.path.join(temp_dir, "Responses")
os.makedirs(responses_dir, exist_ok=True)
download_batched_responses(batches, responses_dir)

Saved responses_headline_clean_part1.jsonl at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Responses
Saved responses_headline_llm_clean_part1.jsonl at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Responses
Saved responses_headline_llm_clean_part2.jsonl at /Users/haya1/Documents/LanguageModel_Labels/headlines_completion/Temp/Embeddings/Responses
